# Long Context and Sliding Window Attention

Standard self-attention computes pairwise scores between every token and every other token in the sequence. This means the computational and memory cost scales as **O(T^2)** where T is the sequence length. At 4K tokens that is manageable, but at 32K, 128K, or 1M tokens the quadratic wall becomes the dominant bottleneck.

**Sliding window attention** is one of the simplest and most effective remedies. Instead of letting every token attend to the full history, each token only looks at the most recent W positions. This reduces per-layer cost to **O(T * W)**, turning quadratic scaling into linear scaling for a fixed window size.

In this notebook we will:
1. Implement a full-attention baseline for comparison.
2. Use `sliding(q, k, v, window)` to compute windowed attention.
3. Visualize attention masks for various window sizes.
4. Compare outputs, computational cost, memory, and information propagation properties.

In [ ]:
%matplotlib inline

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sliding_window import sliding

torch.manual_seed(42)

In [ ]:
# Hyperparameters
embed_dim = 64
seq_len = 32
batch_size = 2

# Random Q, K, V tensors
q = torch.randn(batch_size, seq_len, embed_dim)
k = torch.randn(batch_size, seq_len, embed_dim)
v = torch.randn(batch_size, seq_len, embed_dim)

print(f"Q shape: {q.shape}")
print(f"K shape: {k.shape}")
print(f"V shape: {v.shape}")

## 1. Full Attention Baseline

We start with standard causal self-attention: every token at position t can attend to all positions 0 through t. The attention scores are computed as:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

where M is a causal mask that sets future positions to $-\infty$.

In [ ]:
def full_attention(q, k, v):
    """Standard causal self-attention."""
    T = q.size(1)
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / (d_k ** 0.5)  # (B, T, T)
    # Causal mask: position t can only attend to positions <= t
    causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
    scores = scores.masked_fill(causal_mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

full_out, full_weights = full_attention(q, k, v)
print(f"Full attention output shape: {full_out.shape}")
print(f"Full attention weights shape: {full_weights.shape}")

## 2. Sliding Window Attention

With sliding window attention, each token at position t only attends to positions `max(0, t - window)` through `t`. This bounds the number of keys each query sees to at most `window + 1`, regardless of how long the full sequence is.

In [ ]:
window = 8
sw_out = sliding(q, k, v, window=window)

print(f"Sliding window output shape: {sw_out.shape}")
print(f"Full attention output shape:  {full_out.shape}")
print(f"\nShapes match: {sw_out.shape == full_out.shape}")

Both produce the same output shape `(batch_size, seq_len, embed_dim)`. The difference is *which* keys and values each query position is allowed to see.

## 3. Attention Mask Visualization

Let us visualize the binary attention masks. A cell (t, s) is 1 if token at position t can attend to position s, and 0 otherwise.

In [ ]:
def build_sliding_mask(T, window):
    """Build a binary mask: mask[t, s] = 1 if s in [max(0, t-window), t]."""
    mask = torch.zeros(T, T)
    for t in range(T):
        start = max(0, t - window)
        mask[t, start:t+1] = 1.0
    return mask

def build_causal_mask(T):
    """Build a causal mask: mask[t, s] = 1 if s <= t."""
    return torch.tril(torch.ones(T, T))

In [ ]:
windows_to_show = [4, 8, 16]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Full causal mask
causal = build_causal_mask(seq_len)
axes[0].imshow(causal.numpy(), cmap='Blues', aspect='equal')
axes[0].set_title('Full Causal', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

for i, w in enumerate(windows_to_show):
    mask = build_sliding_mask(seq_len, w)
    axes[i+1].imshow(mask.numpy(), cmap='Blues', aspect='equal')
    axes[i+1].set_title(f'Window = {w}', fontsize=12, fontweight='bold')
    axes[i+1].set_xlabel('Key position')
    axes[i+1].set_ylabel('Query position')

plt.suptitle('Attention Masks: Full Causal vs Sliding Window', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

The full causal mask is a lower triangle: every token sees all previous tokens. Sliding window attention keeps only a narrow diagonal band. As the window grows, the band widens and approaches the full causal mask. At `window >= seq_len - 1` they become identical.

## 4. Comparing Outputs: Full Attention vs Sliding Window

How similar are the outputs of full attention and sliding window attention? We measure **cosine similarity** between the two output tensors at each sequence position.

In [ ]:
def cosine_similarity_per_position(a, b):
    """Compute cosine similarity between a and b at each position. Shape: (B, T)."""
    a_norm = a / (a.norm(dim=-1, keepdim=True) + 1e-8)
    b_norm = b / (b.norm(dim=-1, keepdim=True) + 1e-8)
    return (a_norm * b_norm).sum(dim=-1)

cos_sim = cosine_similarity_per_position(full_out, sw_out)
# Average over batch
cos_sim_avg = cos_sim.mean(dim=0)

print(f"Cosine similarity shape: {cos_sim_avg.shape}")
print(f"Mean cosine similarity: {cos_sim_avg.mean().item():.4f}")
print(f"Min cosine similarity:  {cos_sim_avg.min().item():.4f}")
print(f"Max cosine similarity:  {cos_sim_avg.max().item():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = np.arange(seq_len)
ax.bar(positions, cos_sim_avg.detach().numpy(), color='steelblue', edgecolor='navy', alpha=0.8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Perfect similarity')
ax.set_xlabel('Sequence position', fontsize=12)
ax.set_ylabel('Cosine similarity', fontsize=12)
ax.set_title(f'Full Attention vs Sliding Window (W={window}): Cosine Similarity per Position',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Early positions (where `t <= window`) have identical outputs because the sliding window covers their entire causal history. Later positions diverge because the sliding window drops older keys that the full attention model still sees. The degree of divergence depends on how much information was concentrated in those distant positions.

## 5. Computational Cost Analysis

The dominant cost of attention is the `Q @ K^T` matrix multiplication and the subsequent `weights @ V` multiplication.

- **Full causal attention**: Each of the T query positions attends to on average T/2 keys, giving **O(T^2 * d)** FLOPs.
- **Sliding window attention**: Each query attends to at most (W+1) keys, giving **O(T * W * d)** FLOPs.

We compute the exact FLOP counts for both.

In [ ]:
def flops_full_causal(T, d):
    """FLOPs for full causal attention.
    Q@K^T: T * T * d multiplications (lower triangle averages T/2 per row, but we compute full then mask).
    weights@V: same cost. Total ~ 2 * T * T * d.
    """
    return 2 * T * T * d

def flops_sliding(T, W, d):
    """FLOPs for sliding window attention.
    Each position t attends to min(t+1, W+1) keys.
    Total keys across all positions * d * 2 (for Q@K^T and weights@V).
    """
    total_keys = sum(min(t + 1, W + 1) for t in range(T))
    return 2 * total_keys * d

d = embed_dim
T = seq_len
W = window

flops_f = flops_full_causal(T, d)
flops_s = flops_sliding(T, W, d)

print(f"Full causal FLOPs:    {flops_f:>12,}")
print(f"Sliding (W={W}) FLOPs: {flops_s:>12,}")
print(f"Ratio (full/sliding): {flops_f / flops_s:.2f}x")

## 6. Computation Ratio vs Sequence Length

The savings from sliding window attention grow as the sequence gets longer. Let us plot the FLOP ratio (full / sliding) across a range of sequence lengths for multiple window sizes.

In [ ]:
seq_lengths = np.array([32, 64, 128, 256, 512, 1024, 2048, 4096])
window_sizes = [4, 8, 16, 32]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for w, color in zip(window_sizes, colors):
    ratios = []
    for T in seq_lengths:
        r = flops_full_causal(T, d) / flops_sliding(T, w, d)
        ratios.append(r)
    ax.plot(seq_lengths, ratios, 'o-', color=color, linewidth=2, markersize=6, label=f'W = {w}')

ax.set_xscale('log', base=2)
ax.set_xlabel('Sequence Length (T)', fontsize=12)
ax.set_ylabel('FLOP Ratio (Full / Sliding)', fontsize=12)
ax.set_title('Computational Savings of Sliding Window Attention', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The ratio grows approximately linearly with sequence length (for fixed window), confirming the transition from O(T^2) to O(T*W). For long sequences the savings are dramatic: at T=4096 with W=4, sliding window attention uses roughly 500x fewer FLOPs than full attention.

## 7. Window Size Ablation

How does the window size affect the output? We test a range of window sizes and measure:
- **Output norm**: The L2 norm of the output (stability check).
- **Cosine similarity to full attention**: How close is the windowed output to the full baseline?
- **Effective receptive field**: The number of positions each token can attend to (single layer).

In [ ]:
test_windows = [2, 4, 8, 16, 32]
results = {'window': [], 'mean_norm': [], 'mean_cos_sim': [], 'avg_receptive_field': []}

for w in test_windows:
    out = sliding(q, k, v, window=w)
    
    # Output norm (average over batch and positions)
    norms = out.norm(dim=-1).mean().item()
    
    # Cosine similarity to full attention
    cos = cosine_similarity_per_position(full_out, out).mean().item()
    
    # Effective receptive field: average number of keys per position
    rf = np.mean([min(t + 1, w + 1) for t in range(seq_len)])
    
    results['window'].append(w)
    results['mean_norm'].append(norms)
    results['mean_cos_sim'].append(cos)
    results['avg_receptive_field'].append(rf)

# Print table
print(f"{'Window':>8} {'Mean Norm':>12} {'Cos Sim':>10} {'Avg RF':>10}")
print('-' * 44)
for i in range(len(test_windows)):
    print(f"{results['window'][i]:>8d} {results['mean_norm'][i]:>12.4f} {results['mean_cos_sim'][i]:>10.4f} {results['avg_receptive_field'][i]:>10.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(results['window'], results['mean_norm'], 's-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Window Size', fontsize=11)
axes[0].set_ylabel('Mean Output Norm', fontsize=11)
axes[0].set_title('Output Stability', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(results['window'], results['mean_cos_sim'], 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Window Size', fontsize=11)
axes[1].set_ylabel('Cosine Similarity', fontsize=11)
axes[1].set_title('Similarity to Full Attention', fontsize=12, fontweight='bold')
axes[1].set_ylim(0.5, 1.05)
axes[1].grid(True, alpha=0.3)

axes[2].plot(results['window'], results['avg_receptive_field'], 'D-', color='#2ecc71', linewidth=2, markersize=8)
axes[2].set_xlabel('Window Size', fontsize=11)
axes[2].set_ylabel('Avg Receptive Field', fontsize=11)
axes[2].set_title('Effective Receptive Field (1 Layer)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Window Size Ablation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Key observations:
- **Output norm** remains relatively stable across window sizes, meaning sliding window attention does not introduce instability.
- **Cosine similarity** increases monotonically with window size. At `W=32` (equal to `seq_len`) the output matches full attention exactly.
- **Receptive field** grows linearly with W, capped by the sequence length.

## 8. Attention Pattern Comparison

Let us visualize the actual attention weight matrices for full causal attention vs sliding window attention side by side.

In [ ]:
def sliding_attention_weights(q, k, window):
    """Compute the full T x T attention weight matrix for sliding window.
    Positions outside the window get weight 0."""
    B, T, d = q.shape
    weights = torch.zeros(B, T, T)
    for t in range(T):
        start = max(0, t - window)
        ks = k[:, start:t+1]  # (B, window_size, d)
        scores = q[:, t:t+1] @ ks.transpose(-2, -1) / (d ** 0.5)  # (B, 1, window_size)
        w = torch.softmax(scores, dim=-1)  # (B, 1, window_size)
        weights[:, t, start:t+1] = w.squeeze(1)
    return weights

sw_weights = sliding_attention_weights(q, k, window=8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Use batch element 0
im0 = axes[0].imshow(full_weights[0].detach().numpy(), cmap='viridis', aspect='equal', vmin=0)
axes[0].set_title('Full Causal Attention', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(sw_weights[0].detach().numpy(), cmap='viridis', aspect='equal', vmin=0)
axes[1].set_title('Sliding Window (W=8)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle('Attention Weight Heatmaps', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In the full causal attention heatmap, every row can have non-zero weights across its entire causal history. In the sliding window heatmap, each row only has non-zero weights in a narrow band near the diagonal. The attention probability mass is concentrated into fewer positions, which means each attended key receives a proportionally higher weight.

## 9. Information Propagation Across Layers

A single layer of sliding window attention with window W lets token t see back W positions. But when we **stack** multiple layers, information can propagate further. After L layers, token 0's information can reach token t if:

$$t \leq L \times W$$

Each layer extends the effective receptive field by W positions. So the number of layers needed for token 0 to influence token T is:

$$L = \lceil T / W \rceil$$

This is the key insight behind models like Mistral: with enough layers, even a small window can cover the full context.

In [ ]:
def layers_needed(T, W):
    """Minimum layers for token 0 to influence token T with window W."""
    return int(np.ceil(T / W))

target_positions = [32, 64, 128, 256, 512, 1024, 2048, 4096]
layer_windows = [4, 8, 16, 32, 64]

print(f"{'Target T':>10}", end='')
for w in layer_windows:
    print(f"  W={w:>3}", end='')
print()
print('-' * (10 + 7 * len(layer_windows)))

for T in target_positions:
    print(f"{T:>10}", end='')
    for w in layer_windows:
        print(f"{layers_needed(T, w):>7}", end='')
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

num_layers_range = np.arange(1, 33)
window_sizes_prop = [4, 8, 16, 32]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for w, color in zip(window_sizes_prop, colors):
    receptive_field = num_layers_range * w
    ax.plot(num_layers_range, receptive_field, 'o-', color=color, linewidth=2,
            markersize=4, label=f'W = {w}')

ax.axhline(y=4096, color='black', linestyle=':', alpha=0.5, label='4096 tokens')
ax.set_xlabel('Number of Stacked Layers', fontsize=12)
ax.set_ylabel('Effective Receptive Field (tokens)', fontsize=12)
ax.set_title('Effective Receptive Field vs Number of Layers', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 4500)
plt.tight_layout()
plt.show()

The receptive field grows linearly with the number of layers. With W=32 and 32 layers (a typical transformer depth), the effective receptive field reaches 1024 tokens. A model like Mistral-7B uses W=4096 with 32 layers, giving a theoretical receptive field of 131,072 tokens through information propagation alone.

## 10. Memory Analysis

Attention memory is dominated by storing the attention weight matrix (or the score matrix before softmax).

- **Full attention**: Must store a `T x T` matrix per batch element per head, requiring **O(T^2)** memory.
- **Sliding window**: At most `T x (W+1)` non-zero entries, requiring **O(T * W)** memory.

In practice, the KV cache is often the binding constraint for long-context inference.

In [ ]:
def memory_full(T, d, dtype_bytes=4):
    """Memory for full attention: attention matrix (T x T) + KV cache (2 * T * d)."""
    attn_matrix = T * T * dtype_bytes
    kv_cache = 2 * T * d * dtype_bytes
    return attn_matrix + kv_cache

def memory_sliding(T, W, d, dtype_bytes=4):
    """Memory for sliding window: attention band (T * (W+1)) + KV cache (2 * (W+1) * d)."""
    attn_matrix = T * (W + 1) * dtype_bytes
    # Only need to keep W+1 KV entries in the cache at any time
    kv_cache = 2 * (W + 1) * d * dtype_bytes
    return attn_matrix + kv_cache

seq_lengths_mem = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
d = embed_dim

print(f"{'Seq Len':>10} {'Full (KB)':>12} {'Sliding W=64 (KB)':>20} {'Ratio':>8}")
print('-' * 54)
for T in seq_lengths_mem:
    mem_f = memory_full(T, d) / 1024
    mem_s = memory_sliding(T, 64, d) / 1024
    print(f"{T:>10} {mem_f:>12.1f} {mem_s:>20.1f} {mem_f/mem_s:>8.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

seq_arr = np.array(seq_lengths_mem)
mem_full = np.array([memory_full(T, d) for T in seq_arr]) / (1024 * 1024)  # MB

ax.plot(seq_arr, mem_full, 's-', color='#e74c3c', linewidth=2, markersize=6, label='Full Attention')

for w, color in zip([16, 64, 256], ['#3498db', '#2ecc71', '#9b59b6']):
    mem_sw = np.array([memory_sliding(T, w, d) for T in seq_arr]) / (1024 * 1024)
    ax.plot(seq_arr, mem_sw, 'o-', color=color, linewidth=2, markersize=6, label=f'Sliding W={w}')

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Memory (MB)', fontsize=12)
ax.set_title('Memory Footprint: Full vs Sliding Window Attention', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

Full attention memory grows quadratically (the curve steepens on a log-log plot), while sliding window memory grows linearly. The KV cache advantage is especially important during inference: with a fixed window, you only need to store W key-value pairs regardless of how long the generated sequence becomes.

## 11. When Does Sliding Window Match Full Attention?

Let us verify empirically: for early positions in the sequence (where `t <= W`), sliding window and full causal attention should be identical because the window covers the full causal history.

In [ ]:
for w in [4, 8, 16]:
    out_sw = sliding(q, k, v, window=w)
    # Check positions where the window covers the full causal history
    max_exact = w  # positions 0..w should be exact (t <= w means window covers all of [0..t])
    diff = (full_out[:, :max_exact+1] - out_sw[:, :max_exact+1]).abs().max().item()
    print(f"W={w:>2}: max absolute diff for positions 0..{max_exact}: {diff:.2e}")

The differences are at floating-point precision (around 1e-7), confirming that when the window is large enough to cover a position's full history, the outputs match exactly.

## 12. How Information Degrades with Distance

To see the practical impact of limited context, let us create a scenario where we embed a "signal" at position 0 and measure how much of it survives at later positions under both full and sliding window attention.

In [ ]:
torch.manual_seed(42)

# Create Q, K, V where position 0 has a distinctive signal
q_sig = torch.randn(1, seq_len, embed_dim) * 0.1
k_sig = torch.randn(1, seq_len, embed_dim) * 0.1
v_sig = torch.randn(1, seq_len, embed_dim) * 0.1

# Plant a strong signal at position 0 in the value
signal = torch.ones(1, 1, embed_dim) * 5.0
v_sig[:, 0:1, :] = signal

# Full attention: how much of position 0's value leaks to each position?
full_out_sig, _ = full_attention(q_sig, k_sig, v_sig)
signal_strength_full = (full_out_sig[0] * signal[0, 0]).sum(dim=-1) / (signal[0, 0].norm() * full_out_sig[0].norm(dim=-1) + 1e-8)

# Sliding window
sw_out_sig = sliding(q_sig, k_sig, v_sig, window=8)
signal_strength_sw = (sw_out_sig[0] * signal[0, 0]).sum(dim=-1) / (signal[0, 0].norm() * sw_out_sig[0].norm(dim=-1) + 1e-8)

fig, ax = plt.subplots(figsize=(10, 4))
positions = np.arange(seq_len)
ax.plot(positions, signal_strength_full.detach().numpy(), 'o-', color='#e74c3c',
        linewidth=2, markersize=5, label='Full Causal')
ax.plot(positions, signal_strength_sw.detach().numpy(), 's-', color='#3498db',
        linewidth=2, markersize=5, label='Sliding W=8')
ax.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='Window boundary')
ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('Signal Alignment (cosine)', fontsize=12)
ax.set_title('Signal Propagation from Position 0', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Full causal attention maintains some connection to position 0 across the entire sequence (the signal decays but never reaches zero). Sliding window attention, by contrast, loses the signal entirely once a position falls outside the window. This is the fundamental tradeoff: sliding window saves compute and memory but cannot directly transmit information beyond W positions in a single layer.

---

## Key Insights

**1. Quadratic bottleneck.** Standard self-attention scales as O(T^2) in both compute and memory. This makes long sequences prohibitively expensive.

**2. Linear cost.** Sliding window attention reduces per-layer cost to O(T * W), turning the quadratic scaling into linear scaling for fixed W. The savings grow proportionally with sequence length.

**3. Exact for short histories.** For early positions where the causal history is shorter than the window, sliding window produces identical outputs to full attention. Divergence only occurs at later positions.

**4. Information propagation through depth.** While a single layer limits each token to a window of W, stacking L layers extends the effective receptive field to L * W. A 32-layer model with W=4096 can theoretically propagate information across 131K tokens.

**5. Memory advantage during inference.** The KV cache only needs to store the most recent W key-value pairs, enabling constant-memory inference regardless of generated sequence length.

**6. When to use sliding window:**
- Long sequences where O(T^2) attention is the bottleneck.
- Tasks where most relevant context is local (language modeling, code completion).
- Models with sufficient depth to compensate for limited single-layer receptive field.

**7. When NOT to use sliding window:**
- Tasks requiring direct long-range dependencies in a single layer (e.g., retrieval from the beginning of a long document).
- Very short sequences where the overhead of windowed computation is not justified.
- Situations where the model has few layers and cannot rely on multi-layer propagation.

**8. Real-world usage.** Mistral-7B uses sliding window attention with W=4096. Combined with 32 transformer layers, this provides an effective context of 131K tokens while keeping per-layer attention cost manageable.